# RegistryToolProvider — Dynamic Tool Discovery for Strands Agents

## Overview

This notebook demonstrates the **RegistryToolProvider** — a Strands `ToolProvider` that discovers tools
from AWS Agent Registry via semantic search and injects them into the agent's LLM context automatically.

### The Problem

Every tool given to an LLM costs ~200-500 tokens. An agent with 200 tools burns 40K-100K tokens of context
before the user even asks a question — expensive, slow, and the model gets confused.

### The Solution

The `RegistryToolProvider` searches the Registry with domain keywords before each LLM turn and returns
only the relevant tools. The agent sees 5-10 tools instead of 200.

![image](Images/image.png)

## What You'll Build

1. **Create a Registry** and register 3 public MCP servers via URL synchronization (AWS, Cloudflare, Astro)
2. **Use the RegistryToolProvider** to dynamically discover tools via semantic search
3. **Test domain scoping** — narrow (single server) vs broad (all servers)
4. **Test caching and security controls**
5. **Wire it into a Strands agent** for end-to-end dynamic tool usage
6. **Clean up** all resources

## Prerequisites

- AWS credentials configured
- `boto3>=1.42.87` (for URL sync support)
- `strands-agents`, `httpx`

---

| Information | Details |
|:---|:---|
| Tutorial type | Interactive |
| AgentCore components | Agent Registry, RegistryToolProvider |
| Auth method | IAM |
| Record creation | URL Synchronization (no deployment needed) |
| Example complexity | Intermediate |
| SDK used | boto3 (>=1.42.87), strands-agents |

## Setup

In [ ]:
!pip install -q "boto3>=1.42.87" strands-agents httpx python-dotenv

In [ ]:
import boto3
import json
import time
import sys, os
sys.path.insert(0, os.path.abspath('.'))

from utils import (
    create_registry, seed, wait_for_search_index,
    search, delete_registry, get_cp_client, get_dp_client, REGION,
)

print(f"Region: {REGION}")
print(f"boto3: {boto3.__version__}")

---
## Step 1: Create Registry and Seed MCP Records

We seed 3 inline MCP records with distinct domains (weather, orders, inventory)
plus public MCP servers via URL sync (AWS Knowledge, Find-A-Domain, Peek).
The inline records have 2-3 tools each — small and focused, ideal for demonstrating domain scoping.

- **weather-tools** — `get_current_weather`, `get_weather_forecast`
- **order-management-tools** — `get_order_status`, `list_orders`, `create_order`
- **inventory-tools** — `check_inventory`, `search_products`
- **aws-knowledge-mcp** — (URL sync) AWS documentation search
- **find-a-domain-mcp** — (URL sync) domain name availability
- **peek-mcp** — (URL sync) travel activities and tours

In [ ]:
registry = create_registry(
    name="tool-provider-demo",
    description="Registry for RegistryToolProvider demo — inline MCP records + URL sync",
)
REGISTRY_ID = registry["registryId"]
REGISTRY_ARN = registry["registryArn"]
print(f"Registry ID: {REGISTRY_ID}")

In [ ]:
# Register public MCP servers via URL sync, wait for sync, and approve
seeded = seed(REGISTRY_ID)

print(f"\nSeeded {len(seeded)} records:")
total_tools = 0
for r in seeded:
    print(f"  📦 {r['name']} — {r['tool_count']} tools")
    total_tools += r['tool_count']
print(f"\nTotal tools: {total_tools}")

In [ ]:
# Wait for search index to propagate
wait_for_search_index(REGISTRY_ID, expected_count=len(seeded))

---
## Step 2: RegistryToolProvider — Dynamic Tool Discovery

The `RegistryToolProvider` searches the Registry with domain keywords
and returns only the relevant tools. Different keywords = different tools = bounded LLM context.

In [ ]:
from registry_tool_provider import RegistryToolProvider
import logging
logging.basicConfig(level=logging.INFO)

DP_ENDPOINT = f"https://bedrock-agentcore.{REGION}.amazonaws.com"

provider = RegistryToolProvider(
    registry_ids=[REGISTRY_ID],
    domains=["weather forecast", "order tracking", "inventory stock"],
    region=REGION,
    endpoint_url=DP_ENDPOINT,
    cache_ttl=0,
    required_status="APPROVED",
)

tools = await provider.load_tools()

print(f"🔍 Discovered {len(tools)} tools from {len(provider._domains)} domain(s):\n")
for t in tools:
    print(f"  🔧 {t.tool_name}")
    print(f"     {t.tool_spec['description'][:100]}")
    print()

### Domain scoping — narrow vs broad

This is the core value: different keywords = different tools = bounded context.

In [ ]:
# Narrow: only weather
narrow = RegistryToolProvider(
    registry_ids=[REGISTRY_ID], domains=["weather forecast"],
    region=REGION, endpoint_url=DP_ENDPOINT, cache_ttl=0, required_status="APPROVED",
)
narrow_tools = await narrow.load_tools()
print(f"Narrow ('weather forecast'): {len(narrow_tools)} tools")
for t in narrow_tools: print(f"  - {t.tool_name}")

print()

# Medium: orders only
medium = RegistryToolProvider(
    registry_ids=[REGISTRY_ID], domains=["order tracking shipping"],
    region=REGION, endpoint_url=DP_ENDPOINT, cache_ttl=0, required_status="APPROVED",
)
medium_tools = await medium.load_tools()
print(f"Medium ('order tracking shipping'): {len(medium_tools)} tools")
for t in medium_tools: print(f"  - {t.tool_name}")

print()

# Broad: all domains
broad = RegistryToolProvider(
    registry_ids=[REGISTRY_ID],
    domains=["weather", "orders", "inventory", "shipping", "stock levels"],
    region=REGION, endpoint_url=DP_ENDPOINT, cache_ttl=0, required_status="APPROVED",
)
broad_tools = await broad.load_tools()
print(f"Broad (5 domains): {len(broad_tools)} tools")
for t in broad_tools: print(f"  - {t.tool_name}")

### Caching and consumer lifecycle

In [ ]:
provider._cache_ttl = 300
provider._cache = []
provider._cache_ts = 0

start = time.time()
tools1 = await provider.load_tools()
print(f"First call:  {len(tools1)} tools in {(time.time()-start)*1000:.0f}ms (API call)")

start = time.time()
tools2 = await provider.load_tools()
print(f"Second call: {len(tools2)} tools in {(time.time()-start)*1000:.1f}ms (cache hit)")

provider.add_consumer("agent-1")
provider.add_consumer("agent-2")
print(f"\nConsumers: 2 — cache alive: {bool(provider._cache)}")
provider.remove_consumer("agent-1")
print(f"Consumers: 1 — cache alive: {bool(provider._cache)}")
provider.remove_consumer("agent-2")
print(f"Consumers: 0 — cache alive: {bool(provider._cache)} (cleared!)")

### Security controls

In [ ]:
# HTTPS enforcement
try:
    RegistryToolProvider(registry_ids=["x"], domains=["x"], gateway_url="http://not-secure.com/mcp")
    print("❌ HTTP should have been rejected")
except ValueError as e:
    print(f"✅ HTTPS enforcement: {e}")

# APPROVED-only filtering
draft_provider = RegistryToolProvider(
    registry_ids=[REGISTRY_ID], domains=["weather"],
    region=REGION, endpoint_url=DP_ENDPOINT, cache_ttl=0, required_status="DRAFT",
)
draft_tools = await draft_provider.load_tools()
print(f"✅ Status filtering: APPROVED={len(narrow_tools)} tools, DRAFT={len(draft_tools)} tools")

# Runtime ARN allowlist
restricted = RegistryToolProvider(
    registry_ids=[REGISTRY_ID], domains=["weather"],
    region=REGION, endpoint_url=DP_ENDPOINT, cache_ttl=0,
    allowed_runtime_arns=["arn:aws:bedrock-agentcore:us-east-1:123:runtime/only-this-one"],
)
print("✅ Runtime ARN allowlist configured")

print("\n✅ All security controls working")

---
## Step 3: Wire Into a Strands Agent

End-to-end: the agent discovers tools from Registry automatically.

**Note:** Tool invocation requires a Gateway URL and token. Without them, the agent
discovers tools but calls return an error. The discovery itself is the demo.

In [ ]:
from strands import Agent
from strands.models import BedrockModel

agent_provider = RegistryToolProvider(
    registry_ids=[REGISTRY_ID],
    domains=["weather", "orders", "inventory"],
    region=REGION,
    endpoint_url=DP_ENDPOINT,
    cache_ttl=300,
    required_status="APPROVED",
)

discovered_tools = await agent_provider.load_tools()
print(f"Discovered {len(discovered_tools)} tools from Registry")

agent = Agent(
    model=BedrockModel(model_id="us.anthropic.claude-sonnet-4-20250514-v1:0", region_name="us-east-1"),
    tools=list(discovered_tools),
    system_prompt="You are a helpful assistant. Use the available tools to answer questions. "
                  "If a tool call fails, explain what happened.",
)

print(f"✅ Agent created with {len(discovered_tools)} tools from Registry")
result = agent("What tools do you have available? List them with a brief description of each.")
print(result)

---
## Step 4: Clean Up

In [ ]:
delete_registry(REGISTRY_ID)
print("\n✅ All resources cleaned up")

---
## Summary

| What we tested | Result |
|---|---|
| Registry creation | ✅ IAM auth, no OAuth needed |
| URL sync from public MCP servers | ✅ Auto-populated server + tool schemas |
| Tool discovery via RegistryToolProvider | ✅ Semantic search returns relevant tools |
| Domain scoping | ✅ Different keywords = different tool sets |
| Caching | ✅ First call ~500ms, cached calls <1ms |
| Security controls | ✅ HTTPS enforcement, APPROVED-only filtering |
| Strands agent integration | ✅ Agent discovers tools automatically |